# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-asif1/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes


The queue ranks all 176,738 pages by confidence_weighted_score — a combination of how far a page's CTR falls short of the expected CTR for its position, weighted by impression volume (so high-traffic gaps outrank low-traffic flukes).

**Distribution across reason codes:**
- high_position_low_ctr: 62,586 pages (35.4%)
- low_confidence_signal: 33,532 pages (19.0%)
- monitor_only: 33,148 pages (18.8%)
- on_par_or_above: 24,902 pages (14.1%)
- mid_position_low_ctr: 22,570 pages (12.8%)

**Reason codes and what they mean for action:**
- **high_position_low_ctr** — Page ranks in top_3 or top_10 but under-captures clicks. Action: rewrite title/meta description first; this is the highest-leverage, lowest-effort fix since the page already has the ranking.
- **mid_position_low_ctr** — Page ranks in top_20, underperforming its bucket. Action: metadata review, but pair with a light content/relevance check since page-2 rankings are less stable.
- **low_confidence_signal** — Fewer than 10 impressions. Action: do not act yet; monitor for more data before treating the CTR number as reliable.
- **monitor_only** — Page ranks beyond_20 with a gap. Position itself is the limiting factor here, not metadata — monitor rather than prioritize for a quick metadata fix.
- **on_par_or_above** — Page meets or beats its position's expected CTR. Action: no action needed; candidate for "protect, don't touch."

A human (SEO strategist or content editor) would work down this queue starting from the top, since higher-ranked entries combine the largest gap with the most trustworthy (high-impression) evidence. Notably, over a third of the portfolio (35.4%) falls into the highest-priority category, so the queue also helps triage — not every flagged page can be fixed at once, and rank order tells you where to start.

In [3]:
import pandas as pd
import numpy as np
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
)
df_available = df_march[df_march['gsc_data_available'] == True].copy()

page_level = df_available.groupby(['client_hash_id', 'content_hash_id']).agg(
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean')
).reset_index()

page_level['ctr'] = page_level['total_clicks'] / page_level['total_impressions']

def position_bucket(pos):
    if pos <= 3: return 'top_3'
    elif pos <= 10: return 'top_10'
    elif pos <= 20: return 'top_20'
    else: return 'beyond_20'

page_level['position_bucket'] = page_level['avg_position'].apply(position_bucket)

expected_ctr_by_bucket = page_level.groupby('position_bucket')['ctr'].mean()
page_level['expected_ctr'] = page_level['position_bucket'].map(expected_ctr_by_bucket)
page_level['opportunity_gap'] = page_level['expected_ctr'] - page_level['ctr']
page_level['confidence_weighted_score'] = page_level['opportunity_gap'] * np.log1p(page_level['total_impressions'])

def assign_reason_code_page(row):
    if row['total_impressions'] < 10:
        return 'low_confidence_signal'
    elif row['opportunity_gap'] <= 0:
        return 'on_par_or_above'
    elif row['position_bucket'] in ('top_3', 'top_10'):
        return 'high_position_low_ctr'
    elif row['position_bucket'] == 'top_20':
        return 'mid_position_low_ctr'
    else:
        return 'monitor_only'

page_level['reason_code'] = page_level.apply(assign_reason_code_page, axis=1)
ranked_pages = page_level.sort_values('confidence_weighted_score', ascending=False).reset_index(drop=True)

print("Regenerated. Total ranked pages:", len(ranked_pages))

Regenerated. Total ranked pages: 176738


In [4]:
print("Total ranked pages:", len(ranked_pages))
print(ranked_pages['reason_code'].value_counts())
print()
print(ranked_pages.head(20)[['content_hash_id', 'position_bucket', 'total_impressions', 'ctr', 'reason_code']])

Total ranked pages: 176738
reason_code
high_position_low_ctr    62586
low_confidence_signal    33532
monitor_only             33148
on_par_or_above          24902
mid_position_low_ctr     22570
Name: count, dtype: int64

             content_hash_id position_bucket  total_impressions       ctr  \
0   content_306bc78dff1eb683           top_3              80821  0.000433   
1   content_8d7d99f109e19aa2           top_3             203497  0.001420   
2   content_fc67675904376267           top_3              60172  0.000299   
3   content_c46df0fa61530d86           top_3              70398  0.000597   
4   content_d61fc394d10cba41           top_3              38000  0.000026   
5   content_9ef3d7516483e665           top_3              89229  0.001031   
6   content_b2b85c287474668d           top_3              65304  0.000934   
7   content_7f52754cb72a5991           top_3              43135  0.000649   
8   content_252aa5480bb1f8d7           top_3              66698  0.001124   
9   conte

## 2. Intended use and limits


**Who uses this:** An SEO strategist or content editor doing weekly/monthly triage of a content portfolio, deciding which pages deserve a metadata or content review this cycle.

**Intended use:** As a starting point for prioritization — not a final verdict. The score tells you *where* to look, not definitively *why* a page underperforms or *what exact change* will fix it. It should be used to build a shortlist for manual review, not to trigger automatic changes.

**Where it stops being valid:**
- **Very recent pages** — pages published in the last few days may show low CTR simply because they haven't built search trust/branding yet, not because of a metadata problem. This score does not account for content age.
- **Zero or near-zero impression pages** — already handled via low_confidence_signal, but the threshold (10 impressions) is a judgment call, not a statistically rigorous cutoff.
- **Non-clickable-intent queries** — if a page ranks well for an informational query where users get their answer directly from the search snippet (e.g., a quick definition), low CTR is expected behavior, not a fixable problem. This score cannot distinguish "bad metadata" from "satisfied without a click."
- **Cannibalization cases** — if multiple pages from the same site compete for the same query, one page's low CTR might just mean traffic is going to a sibling page instead. This score treats each page independently and won't catch that.
- **Seasonal/topical decline** — a page's real-world relevance may have dropped for reasons unrelated to its title or metadata (e.g., the topic itself has less search interest right now).

This tool is decision-support, not a decision-maker. It narrows 176,738 pages down to a manageable, ranked shortlist — a human still needs to open each page and confirm the reason before acting.

In [5]:
print("Low-confidence pages (impressions < 10):", (ranked_pages['total_impressions'] < 10).sum())
print("\nImpression distribution for low_confidence_signal pages:")
print(ranked_pages[ranked_pages['reason_code'] == 'low_confidence_signal']['total_impressions'].describe())

Low-confidence pages (impressions < 10): 33532

Impression distribution for low_confidence_signal pages:
count    33532.000000
mean         3.420106
std          2.457639
min          1.000000
25%          1.000000
50%          3.000000
75%          5.000000
max          9.000000
Name: total_impressions, dtype: float64


## 3. Human review + the no-go list


**What a person must check before acting on any flagged page:**
1. Open the page and read the current title and meta description — confirm they're actually generic/unclear, not just numerically flagged.
2. Check publish date — if the page is less than ~30 days old, deprioritize; low CTR may just reflect a page still building trust in search.
3. Search the target query manually — check if a featured snippet, "People Also Ask" box, or a much stronger competing result is absorbing clicks before this page even gets seen.
4. Check for cannibalization — search the site for other pages targeting the same or a very similar query; if one exists, fix ranking consolidation before touching metadata.
5. Confirm the page still reflects current, accurate information — a metadata rewrite on outdated content is a weaker fix than a full content refresh.

**The no-go list — what should never be automated:**
- **Never auto-publish a rewritten title/meta description without human review.** Automated rewrites can misrepresent page content, hurt actual relevance, or trigger unrelated ranking drops.
- **Never treat the confidence_weighted_score as a ground-truth ranking of "worst" pages** — it is a triage signal built from position and impressions only; it has no visibility into content quality, business priority, or brand voice.
- **Never bulk-action the entire high_position_low_ctr group in one pass** — 62,586 pages is far too many to review at once; batch it by highest score first, and treat this as an ongoing queue, not a one-time sweep.
- **Never use this score to make personnel or content-team performance judgments** — it measures page outcomes, not writer or team quality, and doing so risks penalizing writers for structural factors (e.g., competitive queries) outside their control.

In [6]:
# Example: a weekly review batch of the top 50 highest-confidence opportunities
weekly_batch = ranked_pages[ranked_pages['reason_code'] == 'high_position_low_ctr'].head(50)
print(f"Suggested weekly review batch size: {len(weekly_batch)} pages")
print(f"Remaining in this category after this batch: {(ranked_pages['reason_code'] == 'high_position_low_ctr').sum() - 50}")

Suggested weekly review batch size: 50 pages
Remaining in this category after this batch: 62536


## 4. Monitoring / retrain triggers

This baseline is a static snapshot of March 2026 — it will go stale. Here's what should trigger a re-run or a rethink:

**Re-run the scoring (same method, fresh data) when:**
- A new month of data becomes available — expected_ctr benchmarks should be recalculated on recent data, not frozen from March forever, since overall search behavior and portfolio composition shift over time.
- A large batch of pages gets edited/rewritten based on this queue — re-score after 30-60 days to check whether the CTR gap actually closed, not just assume the fix worked.

**Revisit the rule itself (not just re-run it) when:**
- The position_bucket boundaries stop matching how the portfolio's SERP results actually look (e.g., if featured snippets or AI overviews become common enough to change what "top_3 visibility" means for clicks).
- The reason_code distribution shifts dramatically (e.g., if low_confidence_signal balloons to most of the portfolio) — that would suggest something changed upstream in the data (a GSC integration issue, a big traffic drop) rather than genuine content decline.
- A real ML model (like the Week 5/6 Random Forest) starts meaningfully outperforming this baseline (currently it does not — R²=0.005 vs baseline ≈0) — if richer features (e.g., actual title/meta text) become available and a model shows a real, validated lift, the baseline should be reconsidered as the primary tool, not just a comparison point.

**Signal that recommendations have gone stale:**
- A page flagged and "fixed" 60+ days ago that still shows the same reason_code in a fresh scoring run — either the fix didn't work, or the flag was wrong to begin with (e.g., a cannibalization case that wasn't caught).

In [7]:
# Illustrative: how you'd check if a previously-flagged page improved after a fix
# (In practice, you'd re-run the full pipeline on a newer month and compare content_hash_id scores)

example_flagged_ids = ranked_pages[ranked_pages['reason_code'] == 'high_position_low_ctr']['content_hash_id'].head(5).tolist()
print("Example pages flagged this cycle (to re-check next cycle):")
print(example_flagged_ids)
print("\nNext-cycle check: re-run this pipeline on the following month's data,")
print("and confirm these content_hash_ids either dropped out of high_position_low_ctr")
print("(meaning the fix worked) or remain flagged (meaning further investigation is needed).")

Example pages flagged this cycle (to re-check next cycle):
['content_306bc78dff1eb683', 'content_8d7d99f109e19aa2', 'content_fc67675904376267', 'content_c46df0fa61530d86', 'content_d61fc394d10cba41']

Next-cycle check: re-run this pipeline on the following month's data,
and confirm these content_hash_ids either dropped out of high_position_low_ctr
(meaning the fix worked) or remain flagged (meaning further investigation is needed).


## 5. Exports for the paper

Three files are exported to work/outputs/ for reuse in the deployed research paper:

1. **baseline_action_score.csv** — the full ranked queue of 176,738 pages, with position, impressions, CTR, expected CTR, opportunity gap, confidence-weighted score, and reason code. This is the reproducible source of truth behind every claim in the Results and Recommendations sections.
2. **top_20_action_queue.csv** — a digestible example slice (highest-priority pages) to illustrate the output format in the paper without overwhelming a reader with 176K rows.
3. **reason_code_summary.csv** — the distribution of pages across reason codes, ready to turn into a bar chart for the paper (e.g., showing that 35.4% of the portfolio falls into the highest-priority high_position_low_ctr category).

In [8]:
import os
os.makedirs('work/outputs', exist_ok=True)

# 1. Full ranked queue (already have the structure from w04, re-saving here to ensure it exists in this session)
output_cols = ['client_hash_id', 'content_hash_id', 'position_bucket',
               'total_impressions', 'total_clicks', 'avg_position', 'ctr',
               'expected_ctr', 'opportunity_gap', 'confidence_weighted_score', 'reason_code']

ranked_pages[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved: work/outputs/baseline_action_score.csv —", len(ranked_pages), "rows")

# 2. Top-20 action queue (for the paper's Recommendations section — a digestible example)
top_20_export = ranked_pages.head(20)[output_cols]
top_20_export.to_csv('work/outputs/top_20_action_queue.csv', index=False)
print("Saved: work/outputs/top_20_action_queue.csv — 20 rows")

# 3. Summary table (reason code distribution — useful for a chart in the paper)
summary_table = ranked_pages['reason_code'].value_counts().reset_index()
summary_table.columns = ['reason_code', 'page_count']
summary_table.to_csv('work/outputs/reason_code_summary.csv', index=False)
print("Saved: work/outputs/reason_code_summary.csv")
print(summary_table)

Saved: work/outputs/baseline_action_score.csv — 176738 rows
Saved: work/outputs/top_20_action_queue.csv — 20 rows
Saved: work/outputs/reason_code_summary.csv
             reason_code  page_count
0  high_position_low_ctr       62586
1  low_confidence_signal       33532
2           monitor_only       33148
3        on_par_or_above       24902
4   mid_position_low_ctr       22570


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.